# Human Action Recognition: Fine-Tuning CLIP ViT-B/16 with Stable Curves

This notebook trains the **same CLIP ViT-B/16 model** as notebook 03, with settings chosen so the curves are smooth
and the number of settings stays small. Notebook 03 had smooth curves; notebook 04 dropped the two things that made
them smooth (weight averaging and layer-wise learning rates) to save settings, and its CLIP curves became jagged.

**What changed compared with notebook 04**

| Change | Why |
|---|---|
| **EMA (weight averaging)** added back | Each epoch's weights are a running average of the last ~500 steps, not one noisy snapshot. This is the main reason the curves are smooth. |
| **Layer-wise learning rates** added back | The bottom layers (general features) change very little, the top layers adapt more, and the new classifier learns fastest. One rate for every layer was too slow or too disruptive. |
| **Weight decay 0.05** instead of 4e-4 | 0.05 is the usual value for transformers with AdamW. 4e-4 comes from the paper's setup and is far weaker with this optimizer. 4e-4 is still tested as a comparison. |
| **Flip test-time augmentation removed** | It made no difference on the test set (92.2% with and without), so it's one less thing to explain. |
| **Dropout 0.05, 25 epochs, patience 3** | Teacher's values, kept. |

> **On Kaggle:** set **Accelerator → GPU T4 x2** and **Internet → On** (the pretrained CLIP weights are downloaded),
> attach the *Human Action Recognition (HAR) Dataset*, then **Save Version → Save & Run All**.
> The main run takes about **45–60 minutes**. Finished runs are saved, so if the session stops, running again skips them.

# Part 0 — Settings
Six settings. Everything else is fixed inside the training code in Part 3, with a reason next to it.

In [ ]:
EPOCHS = 25            # maximum number of epochs; early stopping usually ends sooner
PATIENCE = 3           # stop after 3 epochs without a better validation accuracy
BASE_LR = 1e-4         # learning rate of the top transformer block (lower blocks get less, the classifier more)
WEIGHT_DECAY = 0.05    # AdamW weight decay: the usual value for transformers
DROPOUT = 0.05         # dropout before the classifier
EMA_DECAY = 0.998      # averages the weights over the last ~500 steps: this smooths the curves

RUN_OPTIONAL = False   # two comparison runs after the main one, about 1 hour more: no EMA, and weight decay 4e-4

# Part 1 — Data

## 1.1 Import libraries
This notebook uses **PyTorch** (a deep-learning framework, like Keras/TensorFlow) and **timm**
("PyTorch Image Models"), a library with hundreds of pretrained image networks, including CLIP.

In [ ]:
import os, gc, glob, copy, math, time, random, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import timm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", message="Detected call of")   # harmless scheduler notice with mixed precision

SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| timm:", timm.__version__)
print("Device: ", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU - turn on the GPU accelerator in Settings!")

## 1.2 Find the dataset

In [ ]:
csv_matches = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)
assert csv_matches, "Dataset not found - click 'Add Input' and attach the Human Action Recognition dataset."
DATA_DIR = os.path.dirname(csv_matches[0])
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")
OUTPUT_DIR = "/kaggle/working"
print("Data folder:", DATA_DIR)

## 1.3 Load the labels
Each action gets a number (`calling → 0`, `clapping → 1`, …), because networks work with numbers.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "Training_set.csv"))
class_names = sorted(df["label"].unique())
label_to_id = {name: i for i, name in enumerate(class_names)}
df["label_id"] = df["label"].map(label_to_id)
NUM_CLASSES = len(class_names)

print(f"{len(df)} labeled images, {NUM_CLASSES} actions")
print(df["label"].value_counts().sort_index().to_string())

## 1.4 Example images

In [ ]:
sample = df.sample(8, random_state=1)
plt.figure(figsize=(14, 6))
for i, (_, row) in enumerate(sample.iterrows()):
    plt.subplot(2, 4, i + 1)
    plt.imshow(Image.open(os.path.join(TRAIN_IMG_DIR, row["filename"])))
    plt.title(row["label"])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 1.5 Find near-duplicate photos (prevent data leakage)
Some photos in the dataset are near-copies of each other (the same picture saved twice, resized or recompressed).
If one copy lands in the **training** set and the other in the **test** set, the model can simply *remember*
the photo, and the test accuracy looks better than it really is. This is called **data leakage**.

To prevent it we give every photo a **fingerprint** with *difference hashing (dHash)*:
1. Shrink the photo to 9 × 8 pixels in grayscale.
2. For each pixel, record whether it is brighter than its right-hand neighbour → **64 yes/no bits**.

Photos whose fingerprints differ in **4 bits or fewer** are put in the same **group**, and a group is never
split across train, validation and test.

The fingerprint is deliberately **cautious**. Besides real copies, it also matches some *different* photos that
share a large plain background (white studio shots, black borders), because plain areas give identical bits.
Keeping two different photos on the same side of the split doesn't leak anything, so that's harmless.
To see how many matches are real copies, we also compare small 32 × 32 colour thumbnails pixel by pixel.

In [ ]:
def dhash(path):
    img = Image.open(path).convert("L").resize((9, 8), Image.BILINEAR)
    px = np.asarray(img, dtype=np.int16)
    return (px[:, 1:] > px[:, :-1]).flatten()          # 8 rows x 8 comparisons = 64 bits

paths = [os.path.join(TRAIN_IMG_DIR, f) for f in df["filename"]]
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    bits = np.array(list(pool.map(dhash, paths)), dtype=np.float32)   # shape: (images, 64)

# Hamming distance = number of bits that differ between two fingerprints
pair_a, pair_b = [], []
for start in range(0, len(bits), 1024):
    chunk = bits[start:start + 1024]
    agree = chunk @ bits.T + (1 - chunk) @ (1 - bits).T
    a, b = np.nonzero(64 - agree <= 4)
    a = a + start
    keep = b > a                                        # count each pair once, skip self-matches
    pair_a.append(a[keep])
    pair_b.append(b[keep])
pair_a, pair_b = np.concatenate(pair_a), np.concatenate(pair_b)

n = len(df)
_, groups = connected_components(coo_matrix((np.ones(len(pair_a)), (pair_a, pair_b)), shape=(n, n)), directed=False)
df["group"] = groups

# How alike are the matched pairs really? Average pixel difference of 32 x 32 colour thumbnails (0-255 scale)
def thumbnail(path):
    return np.asarray(Image.open(path).convert("RGB").resize((32, 32), Image.BILINEAR), dtype=np.float32)
pixel_diff = np.array([np.abs(thumbnail(paths[a]) - thumbnail(paths[b])).mean() for a, b in zip(pair_a, pair_b)])
is_copy = pixel_diff < 10
same_label = df["label"].values[pair_a] == df["label"].values[pair_b]

print(f"Photo pairs with matching fingerprints: {len(pair_a)}")
if len(pair_a):
    print(f"  clearly copies (thumbnails differ by less than 10 out of 255): {is_copy.sum()}"
          f" - same label in {same_label[is_copy].mean() if is_copy.any() else 0:.0%} of them")
    print(f"  other matches (near-copies such as crops, and look-alikes):      {(~is_copy).sum()}")
print(f"Groups: {groups.max() + 1} groups for {n} images")

**Examples.** The first three pairs are copies of the same photo; the differences are watermarks or image quality.
The last pair is a *look-alike*: two different photos that the fingerprint matched because of their plain backgrounds.

In [ ]:
copies = np.where(is_copy)[0]
copies = copies[np.argsort(-pixel_diff[copies])][:3]      # copies with small visible differences
lookalikes = np.where(~is_copy)[0]
lookalikes = lookalikes[np.argsort(pixel_diff[lookalikes])][len(lookalikes) // 2:][:1]   # a typical look-alike
rows = [(k, "copy") for k in copies] + [(k, "look-alike") for k in lookalikes]
if rows:
    plt.figure(figsize=(8, 3.2 * len(rows)))
    for r, (k, kind) in enumerate(rows):
        for j, idx in enumerate((pair_a[k], pair_b[k])):
            plt.subplot(len(rows), 2, 2 * r + j + 1)
            plt.imshow(Image.open(paths[idx]))
            title = f"{df['filename'][idx]} · {df['label'][idx]}"
            if j == 0:
                title = f"{kind.upper()} (pixel difference {pixel_diff[k]:.1f})\n" + title
            plt.title(title, fontsize=9)
            plt.axis("off")
    plt.tight_layout()
    plt.show()

## 1.6 Split into training, validation and test sets
- **Training (80%)** – the model learns from these photos.
- **Validation (10%)** – checked after every epoch to pick the best version of the model.
- **Test (10%)** – used **only once at the very end** to report the final accuracy.

Because the validation set is used to *choose* the best epoch, its score is slightly optimistic.
The untouched **test set** gives an honest final number. *Stratified* keeps the same mix of actions in every set;
*group* keeps near-duplicates together.

(The dataset's own `test` folder has no labels, so it can't measure accuracy. Part 6 explains what we use it for.)

In [ ]:
y_all = df["label_id"].values
all_idx = np.arange(len(df))

split_test = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(split_test.split(all_idx, y_all, groups))

split_val = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=SEED)
tr, va = next(split_val.split(trainval_idx, y_all[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr], trainval_idx[va]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

assert not (set(train_df.group) & set(val_df.group) or set(train_df.group) & set(test_df.group) or set(val_df.group) & set(test_df.group))
print(f"Training:   {len(train_df):>5} images")
print(f"Validation: {len(val_df):>5} images")
print(f"Test:       {len(test_df):>5} images")
print("No near-duplicate group is shared between the sets.")

# Part 2 — Preprocessing

## 2.1 Resize, normalize and augment
**Every image (validation and test):**
- **Resize** to 224 × 224 pixels — the input size CLIP ViT-B/16 was trained on.
- **Normalize** each colour channel with CLIP's own mean and standard deviation, so the inputs look like
  what the network saw during pretraining.

**Training images only — data augmentation** (a new random version every epoch):
- **Random resized crop** — zoom into 40–100% of the photo, so the model sees people at different sizes and positions.
- **Horizontal flip** — a mirrored person is doing the same action.
- **Small rotation** (±10°) and **brightness / contrast** changes (±20%).

Augmentation makes the training set look bigger and more varied, which reduces **overfitting**.

In [ ]:
IMG_SIZE = 224
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10, interpolation=InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

def to_display(t):
    # Undo the normalization so a tensor can be shown as a picture
    mean, std = torch.tensor(CLIP_MEAN).view(3, 1, 1), torch.tensor(CLIP_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

example = Image.open(os.path.join(TRAIN_IMG_DIR, train_df["filename"][0])).convert("RGB")
plt.figure(figsize=(14, 3.5))
plt.subplot(1, 6, 1); plt.imshow(to_display(eval_tf(example))); plt.title("original"); plt.axis("off")
for i in range(5):
    plt.subplot(1, 6, i + 2); plt.imshow(to_display(train_tf(example))); plt.title(f"augmented {i + 1}"); plt.axis("off")
plt.suptitle(f"Action: {train_df['label'][0]}")
plt.tight_layout()
plt.show()

## 2.2 Data loaders
A **Dataset** says how to load one image and its label; a **DataLoader** groups images into **batches of 32**
and loads them in parallel on the CPU while the GPU trains.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() or 2

class HARDataset(Dataset):
    def __init__(self, frame, img_dir, transform, labeled=True):
        self.files = frame["filename"].tolist()
        self.labels = frame["label_id"].tolist() if labeled else [-1] * len(frame)
        self.img_dir, self.transform = img_dir, transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.img_dir, self.files[i])).convert("RGB")
        return self.transform(img), self.labels[i]

def make_loader(frame, img_dir, transform, train, labeled=True):
    return DataLoader(HARDataset(frame, img_dir, transform, labeled),
                      batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)

train_loader = make_loader(train_df, TRAIN_IMG_DIR, train_tf, train=True)
val_loader = make_loader(val_df, TRAIN_IMG_DIR, eval_tf, train=False)
test_loader = make_loader(test_df, TRAIN_IMG_DIR, eval_tf, train=False)

x, y = next(iter(train_loader))
print("Batch of images:", tuple(x.shape), "-> 32 images, 3 colours, 224 x 224 pixels")
print("Batch of labels:", tuple(y.shape))
print("Training steps per epoch:", len(train_loader))

# Part 3 — The Training Recipe (CLIP ViT-B/16)

**Why CLIP ViT-B/16.** It is a Vision Transformer whose image encoder was pretrained (by OpenAI) on 400 million photos
paired with their captions. We keep only the image encoder, add a new 15-neuron output layer, and fine-tune everything on
**our photos and our labels**. The captions were only used to learn the starting weights.

**Fine-tuning recipe.** Each piece has one job:

| Piece | Setting | What it does |
|---|---|---|
| Layer-wise learning rates | top block 1e-4, each lower block × 0.75, classifier × 5 | Protects the general features in the lower layers and lets the top adapt |
| AdamW | weight decay 0.05 (not on biases and norms) | Adaptive step per weight, with weight decay that discourages extreme weights |
| Warm-up, then cosine decay | 1 epoch up, then smooth down to 1% | Stops the random classifier from damaging the pretrained layers at the start |
| Label smoothing | 0.1 | Stops the model being 100% confident on ambiguous photos |
| Gradient clipping | 1.0 | Caps any single very large update |
| Mixed precision (FP16) | on | Faster on the GPU, same accuracy |
| EMA weights | 0.998 | The smoothed copy of the weights is the one measured, saved and tested |
| Early stopping | patience 3 on validation accuracy | Stops when validation stops improving; keeps the best epoch |

**Measured the same way every epoch.** After each epoch, the EMA model is scored on a sample of 2,000 training photos,
the validation set and the test set, all on clean photos. The test set is only reported, never used for a decision.

In [ ]:
import json
MODEL_NAME = "vit_base_patch16_clip_224.openai"
PRETRAINED = True
LAYER_DECAY, HEAD_LR_MULT, LABEL_SMOOTHING, GRAD_CLIP, WARMUP_EPOCHS = 0.75, 5.0, 0.1, 1.0, 1
TRAIN_EVAL_N = 2000    # training photos re-measured with clean images every epoch, for the curves

train_eval_df = train_df.sample(min(TRAIN_EVAL_N, len(train_df)), random_state=SEED).reset_index(drop=True)
train_eval_loader = make_loader(train_eval_df, TRAIN_IMG_DIR, eval_tf, train=False)

def build_clip(dropout):
    try:
        net = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=NUM_CLASSES, drop_rate=dropout)
    except Exception as e:
        raise RuntimeError("Could not download the pretrained CLIP weights. In Kaggle, open Settings, turn Internet ON, "
                           "and run the notebook again.") from e
    return net.to(device)

def layer_id(name, num_blocks):
    if name.startswith(("cls_token", "pos_embed", "patch_embed", "norm_pre")):
        return 0                                  # input embeddings: the lowest layer
    if name.startswith("blocks."):
        return int(name.split(".")[1]) + 1        # transformer blocks 1-12
    return num_blocks + 1                         # final norm and output head: the top

def build_param_groups(model, weight_decay):
    num_blocks = len(model.blocks)
    groups = {}
    for name, p in model.named_parameters():
        lid = layer_id(name, num_blocks)
        is_head = name.startswith("head")
        lr = BASE_LR * HEAD_LR_MULT if is_head else BASE_LR * LAYER_DECAY ** (num_blocks + 1 - lid)
        no_decay = p.ndim <= 1 or name in ("cls_token", "pos_embed")   # biases and norms: no weight decay
        g = groups.setdefault((lid, is_head, no_decay), {
            "params": [], "lr": lr, "weight_decay": 0.0 if no_decay else weight_decay})
        g["params"].append(p)
    return list(groups.values())

class EMA:
    # A running average of the model's weights: smooths the epoch-to-epoch noise
    def __init__(self, model, decay):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))   # average over fewer steps at the very start
        for ema_v, v in zip(self.model.state_dict().values(), model.state_dict().values()):
            if ema_v.dtype.is_floating_point:
                ema_v.mul_(d).add_(v.detach(), alpha=1 - d)
            else:
                ema_v.copy_(v)

@torch.no_grad()
def predict(net, dl):
    net.eval()
    probs, labels = [], []
    for x, y in dl:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            probs.append(net(x.to(device, non_blocking=True)).float().softmax(1).cpu())
        labels.append(y)
    return torch.cat(probs), torch.cat(labels)

def measure(net, dl):
    p, y = predict(net, dl)
    return F.nll_loss(torch.log(p + 1e-9), y).item(), (p.argmax(1) == y).float().mean().item()

def free(net):
    net.cpu(); del net; gc.collect()
    if use_amp:
        torch.cuda.empty_cache()

def train_clip(label, use_ema=True, weight_decay=WEIGHT_DECAY, dropout=DROPOUT):
    # Trains one model with the recipe above. A finished run is saved, and loaded again if the notebook restarts.
    folder = os.path.join(OUTPUT_DIR, "runs", "".join(c if c.isalnum() else "_" for c in label))
    if os.path.exists(os.path.join(folder, "result.json")):
        net = build_clip(dropout)
        net.load_state_dict(torch.load(os.path.join(folder, "weights.pt"), map_location=device))
        with open(os.path.join(folder, "result.json")) as f:
            r = json.load(f)
        print(f"[{label}] loaded the finished run from {folder}")
        return net.eval(), pd.read_csv(os.path.join(folder, "history.csv")), r

    seed_everything(SEED)
    t_start = time.time()
    net = build_clip(dropout)
    opt = torch.optim.AdamW(build_param_groups(net, weight_decay), betas=(0.9, 0.999))
    steps_per_epoch = len(train_loader)
    total, warm = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch
    def lr_factor(s):
        if s < warm:
            return s / warm                                                  # warm-up
        return max(0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))), 0.01)   # cosine decay
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    ema = EMA(net, EMA_DECAY) if use_ema else None
    scored = ema.model if use_ema else net           # the model that is measured, saved and tested
    history, best, best_state, bad, step = [], -1.0, None, 0, 0
    for epoch in range(1, EPOCHS + 1):
        net.train()
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(net(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            sched.step()
            if ema is not None:
                ema.update(net, step)
            step += 1
        row = {"epoch": epoch}
        for s, dl in (("train", train_eval_loader), ("val", val_loader), ("test", test_loader)):
            row[f"{s}_loss"], row[f"{s}_acc"] = measure(scored, dl)
        history.append(row)
        improved = row["val_acc"] > best
        if improved:
            best, bad = row["val_acc"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in scored.state_dict().items()}
        else:
            bad += 1
        print(f"[{label}] epoch {epoch:2d} | train {row['train_acc']:.1%} | val {row['val_acc']:.1%} | "
              f"test {row['test_acc']:.1%}{'  <- best' if improved else ''}")
        if bad >= PATIENCE:
            print(f"[{label}] early stopping: no better validation accuracy for {PATIENCE} epochs")
            break
    scored.load_state_dict(best_state)

    h = pd.DataFrame(history)
    b = h.loc[h["val_acc"].idxmax()]
    bumps = h["val_acc"].diff().abs().dropna() * 100
    result = {"model": label, "EMA": use_ema, "weight decay": weight_decay, "dropout": dropout,
              "epochs run": len(h), "best epoch": int(b["epoch"]),
              "train acc": float(b["train_acc"]), "val acc": float(b["val_acc"]), "test acc": float(b["test_acc"]),
              "train-val gap": float(b["train_acc"] - b["val_acc"]),
              "bumpiness (avg val change, pts)": round(float(bumps.mean()), 2),
              "validation drops": int((h["val_acc"].diff() < 0).sum()),
              "minutes": round((time.time() - t_start) / 60, 1)}
    os.makedirs(folder, exist_ok=True)
    torch.save(scored.state_dict(), os.path.join(folder, "weights.pt"))
    h.to_csv(os.path.join(folder, "history.csv"), index=False)
    with open(os.path.join(folder, "result.json"), "w") as f:
        json.dump(result, f)
    print(f"[{label}] best epoch {result['best epoch']}: validation {result['val acc']:.1%}, test {result['test acc']:.1%}")
    return scored, h, result

# Part 4 — Training the Main Model
This is the model to present. It runs the recipe above with all six settings.

In [ ]:
results, histories = [], {}
final_model, h_main, r_main = train_clip("CLIP ViT-B/16 (main)")
results.append(r_main)
histories[r_main["model"]] = h_main

# Part 5 — Results and Curves
## 5.1 Results
**Bumpiness** is the average change in validation accuracy from one epoch to the next, in percentage points.
Smaller means smoother. **Validation drops** counts the epochs where validation accuracy fell.

In [ ]:
def plot_run(h, title):
    best = int(h.loc[h["val_acc"].idxmax(), "epoch"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    for ax, metric, name in ((axes[0], "acc", "Accuracy (%)"), (axes[1], "loss", "Loss")):
        for s, style, lab in (("train", "o-", "train"), ("val", "s-", "validation"), ("test", "^--", "test")):
            vals = h[f"{s}_{metric}"] * (100 if metric == "acc" else 1)
            ax.plot(h["epoch"], vals, style, ms=4, label=lab)
        ax.axvline(best, color="gray", linestyle="--", linewidth=1, label=f"best epoch {best}")
        ax.set_xlabel("Epoch"); ax.set_ylabel(name); ax.grid(alpha=0.3); ax.legend(fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.show()

res = pd.DataFrame(results)
display(res.style.format({"train acc": "{:.1%}", "val acc": "{:.1%}", "test acc": "{:.1%}",
                          "train-val gap": "{:+.1%}", "bumpiness (avg val change, pts)": "{:.2f}"}))
plot_run(h_main, "CLIP ViT-B/16: train, validation and test measured the same way after every epoch")

## 5.2 Comparison runs (optional)
Each run changes **one thing** from the main run, so its effect is clear. Turn them on with `RUN_OPTIONAL = True` in Part 0.
- **No EMA:** the same model without weight averaging. If its curve is jagged and the main run's is smooth, that proves the cause.
- **Weight decay 4e-4:** the value from the paper's setup, for comparison with 0.05.

In [ ]:
if RUN_OPTIONAL:
    for label, kw in (("no EMA (everything else the same)", {"use_ema": False}),
                       ("weight decay 4e-4 (paper value)", {"weight_decay": 4e-4})):
        net_x, h_x, r_x = train_clip(label, **kw)
        results.append(r_x)
        histories[label] = h_x
        free(net_x)
    res = pd.DataFrame(results)
    display(res.style.format({"train acc": "{:.1%}", "val acc": "{:.1%}", "test acc": "{:.1%}",
                              "train-val gap": "{:+.1%}", "bumpiness (avg val change, pts)": "{:.2f}"}))
    fig, ax = plt.subplots(figsize=(10, 4.8))
    for label, h in histories.items():
        bump = h["val_acc"].diff().abs().mean() * 100
        ax.plot(h["epoch"], h["val_acc"] * 100, "o-", ms=3, label=f"{label} (bumpiness {bump:.2f})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Validation accuracy (%)"); ax.grid(alpha=0.3); ax.legend(fontsize=9)
    ax.set_title("Validation accuracy: the same model with and without weight averaging")
    plt.tight_layout(); plt.show()
    res.to_csv(os.path.join(OUTPUT_DIR, "clip_comparison.csv"), index=False)
else:
    print("Comparison runs skipped (RUN_OPTIONAL = False).")

# Part 6 — Per-Action Results and Mistakes
Measured on the **test set** with the main model. The check at the start confirms it matches the result in Part 4.

In [ ]:
test_probs, test_labels = predict(final_model, test_loader)
y_true, y_pred = test_labels.numpy(), test_probs.argmax(1).numpy()
assert abs((y_pred == y_true).mean() - r_main["test acc"]) < 1e-3, "test accuracy does not match the training run"
print(f"Test accuracy: {(y_pred == y_true).mean():.1%}")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=60, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (test set)")
fig.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

pairs = sorted(((cm[i, j], i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j), reverse=True)[:5]
print("Most common mix-ups:")
for n, i, j in pairs:
    print(f"  {class_names[i]:>20} predicted as {class_names[j]:<20} {n} times")

# Part 7 — Explainable AI
- **Occlusion maps:** a grey square slides over the photo; red areas are where covering the photo makes the prediction drop most.
- **Attention rollout:** combines the transformer's attention across all 12 layers, showing what the [CLS] token looks at.

In [ ]:
xai_model = final_model
pred, true = y_pred, y_true
MEAN = torch.tensor(CLIP_MEAN).view(3, 1, 1)
STD = torch.tensor(CLIP_STD).view(3, 1, 1)
def to_img(t):
    return (t * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

@torch.no_grad()
def occlusion_map(x, target, size=32, stride=16):
    xs, pos = [], []
    for top in range(0, IMG_SIZE - size + 1, stride):
        for left in range(0, IMG_SIZE - size + 1, stride):
            xo = x.clone(); xo[:, top:top + size, left:left + size] = 0    # 0 = the average colour after normalization
            xs.append(xo); pos.append((top, left))
    base = xai_model(x[None].to(device)).float().softmax(1)[0, target].item()
    drops = []
    for i in range(0, len(xs), 64):
        p = xai_model(torch.stack(xs[i:i + 64]).to(device)).float().softmax(1)[:, target].cpu()
        drops.append(base - p)
    drops = torch.cat(drops)
    heat, count = torch.zeros(IMG_SIZE, IMG_SIZE), torch.zeros(IMG_SIZE, IMG_SIZE)
    for d, (top, left) in zip(drops, pos):
        heat[top:top + size, left:left + size] += d; count[top:top + size, left:left + size] += 1
    return (heat / count.clamp(min=1)).clamp(min=0).numpy()

def load_test_photo(i):
    return eval_tf(Image.open(os.path.join(TRAIN_IMG_DIR, test_df["filename"][i])).convert("RGB"))

def show_maps(rows, title):
    fig, axes = plt.subplots(len(rows), 3, figsize=(11, 3.6 * len(rows)), squeeze=False)
    for r, i in enumerate(rows):
        x = load_test_photo(i)
        p = int(pred[i]); heat = occlusion_map(x, p)
        axes[r, 0].imshow(to_img(x)); axes[r, 0].set_title(f"true: {class_names[int(true[i])]}", fontsize=10)
        axes[r, 1].imshow(to_img(x)); axes[r, 1].imshow(heat, cmap="jet", alpha=0.5)
        axes[r, 1].set_title(f"predicted: {class_names[p]} ({test_probs[i, p]:.0%})", fontsize=10)
        axes[r, 2].imshow(heat, cmap="jet"); axes[r, 2].set_title("occlusion map (red = important)", fontsize=10)
        for a in axes[r]:
            a.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

rng = np.random.default_rng(SEED)
correct = np.where(pred == true)[0]
wrong = np.where(pred != true)[0]
show_maps(rng.choice(correct, size=min(6, len(correct)), replace=False), "Correct predictions: where the model looks")

In [ ]:
def attention_rollout(x):
    # Combines the [CLS] attention of all transformer layers (Abnar & Zuidema, 2020)
    store, hooks = [], []
    for blk in xai_model.blocks:
        blk.attn.fused_attn = False
        hooks.append(blk.attn.attn_drop.register_forward_hook(lambda m, i, o: store.append(i[0].float().mean(1)[0].cpu())))
    with torch.no_grad():
        xai_model(x[None].to(device))
    for h in hooks:
        h.remove()
    roll = torch.eye(store[0].shape[-1])
    for a in store:
        a = a + torch.eye(a.shape[-1]); a = a / a.sum(-1, keepdim=True)
        roll = a @ roll
    n = int(round((roll.shape[-1] - 1) ** 0.5))
    m = roll[0, 1:1 + n * n].reshape(1, 1, n, n)
    return F.interpolate(m, size=(IMG_SIZE, IMG_SIZE), mode="bilinear")[0, 0].numpy()

sample = rng.choice(correct, size=min(6, len(correct)), replace=False)
fig, axes = plt.subplots(2, len(sample), figsize=(3.2 * len(sample), 6.6), squeeze=False)
for c, i in enumerate(sample):
    x = load_test_photo(i)
    axes[0, c].imshow(to_img(x)); axes[0, c].set_title(class_names[int(pred[i])], fontsize=10)
    axes[1, c].imshow(to_img(x)); axes[1, c].imshow(attention_rollout(x), cmap="jet", alpha=0.5)
    axes[0, c].axis("off"); axes[1, c].axis("off")
fig.suptitle("Attention rollout: what the transformer attends to (bottom row)"); plt.tight_layout(); plt.show()

examples = [k for _, i, j in pairs[:3] for k in wrong if true[k] == i and pred[k] == j][:6]
if examples:
    show_maps(examples, "Mistakes: what the model looked at when it got it wrong")
conf = test_probs.max(1).values.numpy()
plt.figure(figsize=(7, 3.5))
plt.hist(conf[pred == true], bins=20, range=(0, 1), alpha=0.7, label="correct")
plt.hist(conf[pred != true], bins=20, range=(0, 1), alpha=0.7, label="wrong")
plt.xlabel("Confidence of the prediction"); plt.ylabel("Test photos"); plt.legend()
plt.title("Wrong predictions are usually less confident"); plt.tight_layout(); plt.show()

# Summary
- **Settings (six):** 25 epochs, patience 3, base learning rate 1e-4, weight decay 0.05, dropout 0.05, EMA 0.998.
- **Fixed in the code, with reasons:** layer-wise learning rates (×0.75 per layer), warm-up then cosine, label smoothing 0.1,
  gradient clipping 1.0, batch 32, AdamW, FP16.
- **Smooth curves:** EMA averages the weights, so each epoch changes the model a little. The bumpiness number in Part 5 measures it.
- **Results:** Part 4 (main run), Part 5 (comparison runs), Part 6 (per action), Part 7 (explainable AI).